In [1]:
import pandas as pd
import time
import logging
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import re

logging.basicConfig(level=logging.INFO)

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

def esperar_pagina(driver, timeout=10):
    try:
        WebDriverWait(driver, timeout).until(
            EC.any_of(
                EC.presence_of_element_located((By.CLASS_NAME, "break-wrap")),
                EC.presence_of_element_located((By.CLASS_NAME, "ng-tns-c119-0  ng-star-inserted"))
            )
        )
    except Exception as e:
        logging.warning(f"Timeout esperando página: {e}")

def extrair_temporal_coverage(html):
    soup = BeautifulSoup(html, "html.parser")
    spans_visiveis = soup.find_all("span", class_="break-wrap")
    for span in spans_visiveis:
        if span.has_attr("hidden"):
            continue
        for p in span.find_all("p"):
            if "Temporal Coverage:" in p.get_text():
                return p.get_text().replace("Temporal Coverage:", "").strip()
            
    li_tags = soup.find_all("li", class_="ng-tns-c119-0 ng-star-inserted")
    for li in li_tags:
        texto = li.get_text(strip=True)
        # procurar dois anos com espaço ou hífen entre eles
        match = re.match(r"(\d{4})\s*[-–]\s*(\d{4})", texto)
        if match:
            return f"{match.group(1)} to {match.group(2)}"
        
    return None

# Início do processo
driver = webdriver.Chrome(options=options)
df = pd.read_csv("./candidatos_schema_matching_worldbank.csv")
resultados = []

for _, row in df.iterrows():
    dataset_id = str(row["dataset_unique_id"]).zfill(7)
    nome = row["nome_dataset"]
    url = f"https://datacatalog.worldbank.org/search/dataset/{dataset_id}"

    try:
        driver.get(url)
        esperar_pagina(driver)
        html = driver.page_source
        temporal = extrair_temporal_coverage(html)

        resultados.append({
            "dataset_unique_id": dataset_id,
            "nome_dataset": nome,
            "temporal_coverage": temporal
        })

    except Exception as e:
        logging.warning(f"Erro ao processar {dataset_id} - {nome}: {e}")
        resultados.append({
            "dataset_unique_id": dataset_id,
            "nome_dataset": nome,
            "temporal_coverage": None
        })

    time.sleep(0.5)  # pausa leve para respeitar o site

driver.quit()
pd.DataFrame(resultados).to_csv("./candidatos_worldbank_temporal_coverage.csv", index=False)